# 🚀 Local Question Paraphraser Training (Google Colab Version)

이 노트북은 할루시네이션 탐지 시스템의 **Layer 3 (SAC³)**에 사용되는 한국어 질문 패러프레이징 모델을 구글 코랩의 GPU(T4 등)를 활용하여 파인튜닝하는 스크립트입니다.

## 작업 순서
1. **GPU 활성화**: 코랩 상단 메뉴 `런타임` -> `런타임 유형 변경` -> `T4 GPU` 선택
2. **전체 실행**: 모든 셀을 차례대로 실행
3. **다운로드**: 학습 완료 후 생성되는 `local-qwen-paraphraser.zip` 파일을 다운로드하여 로컬 프로젝트 폴더(`Project/Sever/`)에 압축 해제

In [ ]:
# 1. 필요한 패키지 설치
!pip install -q transformers datasets accelerate evaluate

In [ ]:
# 2. 한국어 패러프레이즈 데이터셋(ParaKQC) 다운로드 및 파싱
import urllib.request
from datasets import Dataset

print("Downloading KakaoBrain ParaKQC dataset...")
url = "https://raw.githubusercontent.com/warnikchow/paraKQC/master/data/paraKQC_v1.txt"

req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
lines = []
with urllib.request.urlopen(req) as response:
    for line in response:
        lines.append(line.decode('utf-8').strip())

# 10개 문장씩 묶어 올바른 패러프레이즈 그룹 형성 (Semantic grouping 버그 해결)
label_to_sentences = {}
valid_lines = [line for line in lines if line]
for index, line in enumerate(valid_lines):
    parts = line.split('\t')
    if len(parts) == 3:
        idx, label, sentence = parts
        group_id = index // 10
        if group_id not in label_to_sentences:
            label_to_sentences[group_id] = []
        label_to_sentences[group_id].append(sentence)

num_sentences = sum(len(s) for s in label_to_sentences.values())
print(f"Loaded {num_sentences} sentences across {len(label_to_sentences)} unique paraphrase groups.")

In [ ]:
# 3. 학습을 위한 문장 쌍(Pairs) 생성 및 데이터셋 분할
print("Generating sentence pairs...")
pairs = []
for group_id, sentences in label_to_sentences.items():
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                pairs.append({"input": sentences[i], "target": sentences[j]})

import random
random.seed(42)
random.shuffle(pairs)

# 5000쌍 추출 (VRAM 한계 및 빠른 학습 속도 유지)
pairs = pairs[:5000]
print(f"Generated {len(pairs)} training pairs.")

split_idx = 4500
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

train_dataset = Dataset.from_list(train_pairs)
val_dataset = Dataset.from_list(val_pairs)

In [ ]:
# 4. Qwen-0.5B 토크나이저 및 모델 로드 (bfloat16 정밀도 지원)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading tokenizer and model: {base_model}...")
tokenizer = AutoTokenizer.from_pretrained(base_model)

# 코랩 GPU 환경이므로 네이티브 bfloat16 정밀도로 안전하게 로드
model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.bfloat16)
model.to("cuda")
print("Model loaded on GPU successfully!")

In [ ]:
# 5. 전처리(Preprocessing) 함수 정의 및 매핑
system_prompt = "You are a question paraphrasing expert. Generate exactly 1 semantic variation of the given question."

def preprocess_function(examples):
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for inp, tgt in zip(examples["input"], examples["target"]):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: \"{inp}\""},
            {"role": "assistant", "content": tgt}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        tokenized = tokenizer(text, max_length=256, truncation=True)
        
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        
        # 프롬프트 부분은 Loss 계산에서 제외하기 위해 -100 처리
        prompt_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: \"{inp}\""}
        ]
        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        prompt_tokenized = tokenizer(prompt_text)
        prompt_len = len(prompt_tokenized["input_ids"])

        labels = [-100] * prompt_len + input_ids[prompt_len:]
        
        # 패딩 처리
        padding_len = 256 - len(input_ids)
        if padding_len > 0:
            input_ids = input_ids + [tokenizer.pad_token_id] * padding_len
            attention_mask = attention_mask + [0] * padding_len
            labels = labels + [-100] * padding_len
        else:
            input_ids = input_ids[:256]
            attention_mask = attention_mask[:256]
            labels = labels[:256]

        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)

    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": batch_labels
    }

print("Preprocessing datasets...")
train_tokenized = train_dataset.map(preprocess_function, batched=True, remove_columns=["input", "target"])
val_tokenized = val_dataset.map(preprocess_function, batched=True, remove_columns=["input", "target"])

In [ ]:
# 6. 학습 파라미터 정의 및 SFT 학습 진행
from transformers import TrainingArguments, Trainer

model_dir = "local-qwen-paraphraser"
training_args = TrainingArguments(
    output_dir=model_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,   # 코랩 GPU VRAM(16GB)이 넉넉하므로 배치 사이즈 상향
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,   # Effective batch size = 8
    bf16=True,                       # 네이티브 BFloat16 SFT 학습 수행 (가장 안정적)
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,              # 0.5B 모델이므로 1 에포크로 충분히 파라프레이징 특화 가능
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
)

print("Training Qwen paraphraser model...")
trainer.train()

print("Saving the best model locally...")
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)
print(f"Model saved to {model_dir}/")

In [ ]:
# 7. 다운로드용 압축 파일 생성
!zip -r local-qwen-paraphraser.zip local-qwen-paraphraser
print("==================================================")
print("🎉 압축이 완료되었습니다!")
print("코랩 좌측 메뉴의 폴더 아이콘을 클릭한 뒤, 'local-qwen-paraphraser.zip'을 다운로드해 주세요.")
print("==================================================")